# Business Entity Resolution — Comprehensive EDA

**Scope:** Full-population analysis of all training data (~2.2 M S1 + ~5.0 M S2 + ~5.3 M S3 + ~2.2 M ground-truth rows).  
**No sampling** — every row counts.

---

## Table of Contents

0. [Setup & Data Loading](#0)
1. [Referential Integrity (Gate Check)](#1)
2. [Row-Position / ID Leakage Detection](#2)
3. [Label Structure: Singleton Rate & Match-Count Distribution](#3)
4. [Country-Field Trustworthiness](#4)
5. [Legal-Suffix Inventory per Country](#5)
6. [Address Component Order & Postal-Code Presence](#6)
7. [Junk Characters & Encoding Artifacts](#7)
8. [Reverse-Engineering the Corruption Function](#8)
9. [Within-Source Name / Address Collision Risk](#9)
10. [Additional Observations](#10)

<a id='0'></a>
## 0. Setup & Data Loading

In [1]:
import os, re, sys, time, unicodedata
from pathlib import Path
from collections import Counter, defaultdict
from itertools import islice

import numpy as np
import pandas as pd

try:
    from rapidfuzz import fuzz as rfuzz
    from rapidfuzz.distance import Levenshtein as lev
    HAS_RAPIDFUZZ = True
except ImportError:
    HAS_RAPIDFUZZ = False

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)

# --------------- paths ---------------
BASE = Path(os.getcwd())
# Walk up until we find the dataset/ dir
while not (BASE / 'dataset').exists() and BASE != BASE.parent:
    BASE = BASE.parent
TRAIN = BASE / 'dataset' / 'train'

print(f'Base dir : {BASE}')
print(f'Train dir: {TRAIN}')
for f in sorted(TRAIN.glob('*.tsv')):
    mb = f.stat().st_size / 1024 / 1024
    print(f'  {f.name:40s} {mb:>8.1f} MB')

Base dir : /Users/pranjal/Projects/Amazon-ML/Amazon-ML-2026
Train dir: /Users/pranjal/Projects/Amazon-ML/Amazon-ML-2026/dataset/train
  train_ground_truth.tsv                      121.1 MB
  train_source1.tsv                           200.3 MB
  train_source2.tsv                           466.6 MB
  train_source3.tsv                           480.4 MB


In [2]:
print('Loading source files (this may take a minute)...')
t0 = time.time()

s1 = pd.read_csv(TRAIN / 'train_source1.tsv', sep='\t', dtype=str)
print(f'  S1: {len(s1):>12,} rows  ({time.time()-t0:.1f}s)')

t1 = time.time()
s2 = pd.read_csv(TRAIN / 'train_source2.tsv', sep='\t', dtype=str)
print(f'  S2: {len(s2):>12,} rows  ({time.time()-t1:.1f}s)')

t2 = time.time()
s3 = pd.read_csv(TRAIN / 'train_source3.tsv', sep='\t', dtype=str)
print(f'  S3: {len(s3):>12,} rows  ({time.time()-t2:.1f}s)')

t3 = time.time()
gt = pd.read_csv(TRAIN / 'train_ground_truth.tsv', sep='\t', dtype=str)
print(f'  GT: {len(gt):>12,} rows  ({time.time()-t3:.1f}s)')

print(f'\nTotal load time: {time.time()-t0:.1f}s')
print(f'\nColumn schemas:')
for name, df in [('S1', s1), ('S2', s2), ('S3', s3), ('GT', gt)]:
    print(f'  {name}: {list(df.columns)}')

Loading source files (this may take a minute)...
  S1:    2,206,821 rows  (4.0s)
  S2:    5,034,616 rows  (10.6s)
  S3:    5,285,603 rows  (10.8s)
  GT:    2,206,821 rows  (2.4s)

Total load time: 27.8s

Column schemas:
  S1: ['entity_id', 'business_name', 'business_address', 'country']
  S2: ['entity_id', 'business_name', 'business_address', 'country']
  S3: ['entity_id', 'business_name', 'business_address', 'country']
  GT: ['source1_entity_id', 'matched_entity_ids']


In [3]:
# Preview data
print('=== S1 head ===')
display(s1.head(5)) if hasattr(__builtins__, '__IPYTHON__') or 'IPython' in sys.modules else print(s1.head(5).to_string())
print('\n=== S2 head ===')
print(s2.head(5).to_string())
print('\n=== S3 head ===')
print(s3.head(5).to_string())
print('\n=== GT head ===')
print(gt.head(10).to_string())

=== S1 head ===


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West Bengal",India



=== S2 head ===
      entity_id                    business_name                                  business_address country
0  S2-166376419  राम मार्केटिंग प्राइवेट लिमिटेड      KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi   India
1  S2-764573417     -- Holloway Peak Inc Seafood                         105 ELM ST, MORGANTON, NC      US
2  S2-639257739         आदित्य प्रॉपर्टीज एलएलपी  G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh   India
3  S2-163963287                       Summit Inc             GREENSBORO, NC, 19 1/2 STARDUST TRAIL      US
4   S2-49942811     Delta Tetlecommunication Inc                   914 PIERPONT AVE, CLEVELAND, OH      US

=== S3 head ===
      entity_id                                business_name                                                                             business_address country
0  S3-202863386                           wilfordhancock.com                                                                  Mack Rd, Haltom City, Texas      US


<a id='1'></a>
## 1. Referential Integrity (Gate Check)

Before trusting any analysis, verify:
- Every `source1_entity_id` in GT exists in S1
- Every matched S2/S3 ID exists in S2/S3 respectively
- Every S1 entity appears exactly once in GT
- No duplicate `entity_id` within any source
- No exact-duplicate rows within any source

In [4]:
print('=' * 70)
print('REFERENTIAL INTEGRITY CHECKS')
print('=' * 70)

all_ok = True

# 1a. Unique entity_id within each source
for name, df in [('S1', s1), ('S2', s2), ('S3', s3)]:
    n_dup = df['entity_id'].duplicated().sum()
    status = 'PASS' if n_dup == 0 else 'FAIL'
    if n_dup > 0:
        all_ok = False
    print(f'[{status}] {name} entity_id uniqueness: {n_dup} duplicates out of {len(df):,}')

# 1b. S1 coverage in GT (every S1 id appears in GT, and vice versa)
s1_ids = set(s1['entity_id'])
gt_s1_ids = set(gt['source1_entity_id'])
missing_from_gt = s1_ids - gt_s1_ids
extra_in_gt = gt_s1_ids - s1_ids
print(f'\n[{"PASS" if len(missing_from_gt)==0 else "FAIL"}] S1 ids missing from GT: {len(missing_from_gt):,}')
print(f'[{"PASS" if len(extra_in_gt)==0 else "FAIL"}] GT ids not in S1: {len(extra_in_gt):,}')
if missing_from_gt:
    all_ok = False
    print(f'  Examples of missing: {list(islice(missing_from_gt, 5))}')
if extra_in_gt:
    all_ok = False
    print(f'  Examples of extra: {list(islice(extra_in_gt, 5))}')

# 1c. Every S1 entity appears exactly once in GT
gt_s1_dup = gt['source1_entity_id'].duplicated().sum()
print(f'[{"PASS" if gt_s1_dup==0 else "FAIL"}] GT source1_entity_id uniqueness: {gt_s1_dup} duplicates')
if gt_s1_dup > 0:
    all_ok = False

# 1d. Every matched S2/S3 ID exists in its source
print('\nChecking matched_entity_ids referential integrity...')
s2_ids = set(s2['entity_id'])
s3_ids = set(s3['entity_id'])

bad_s2, bad_s3, bad_other = 0, 0, 0
total_matched = 0
for mids in gt['matched_entity_ids'].dropna():
    for mid in str(mids).split(','):
        mid = mid.strip()
        if not mid:
            continue
        total_matched += 1
        if mid.startswith('S2-'):
            if mid not in s2_ids:
                bad_s2 += 1
        elif mid.startswith('S3-'):
            if mid not in s3_ids:
                bad_s3 += 1
        else:
            bad_other += 1

print(f'  Total matched IDs referenced: {total_matched:,}')
print(f'[{"PASS" if bad_s2==0 else "FAIL"}]   S2 IDs not found in S2 file: {bad_s2}')
print(f'[{"PASS" if bad_s3==0 else "FAIL"}]   S3 IDs not found in S3 file: {bad_s3}')
print(f'[{"PASS" if bad_other==0 else "FAIL"}]   IDs with unexpected prefix: {bad_other}')
if bad_s2 or bad_s3 or bad_other:
    all_ok = False

# 1e. Exact duplicate rows
for name, df in [('S1', s1), ('S2', s2), ('S3', s3)]:
    n_exact_dup = df.duplicated().sum()
    print(f'[{"PASS" if n_exact_dup==0 else "WARN"}] {name} exact-duplicate rows: {n_exact_dup}')

# 1f. Null / missing value audit
print('\n--- Missing value counts per source ---')
for name, df in [('S1', s1), ('S2', s2), ('S3', s3)]:
    print(f'\n  {name}:')
    for col in df.columns:
        n_null = df[col].isna().sum()
        n_empty = (df[col].fillna('') == '').sum()
        print(f'    {col:25s}  null={n_null:>8,}  empty_str={n_empty:>8,}  ({n_null/len(df)*100:.3f}%)')

print(f'\n{"=" * 70}')
print(f'OVERALL: {"ALL CHECKS PASSED" if all_ok else "SOME CHECKS FAILED — INVESTIGATE BEFORE PROCEEDING"}')
print(f'{"=" * 70}')

REFERENTIAL INTEGRITY CHECKS
[PASS] S1 entity_id uniqueness: 0 duplicates out of 2,206,821
[PASS] S2 entity_id uniqueness: 0 duplicates out of 5,034,616
[PASS] S3 entity_id uniqueness: 0 duplicates out of 5,285,603

[PASS] S1 ids missing from GT: 0
[PASS] GT ids not in S1: 0
[PASS] GT source1_entity_id uniqueness: 0 duplicates

Checking matched_entity_ids referential integrity...
  Total matched IDs referenced: 7,638,365
[PASS]   S2 IDs not found in S2 file: 0
[PASS]   S3 IDs not found in S3 file: 0
[PASS]   IDs with unexpected prefix: 0
[PASS] S1 exact-duplicate rows: 0
[PASS] S2 exact-duplicate rows: 0
[PASS] S3 exact-duplicate rows: 0

--- Missing value counts per source ---

  S1:
    entity_id                  null=       0  empty_str=       0  (0.000%)
    business_name              null=       0  empty_str=       0  (0.000%)
    business_address           null=       0  empty_str=       0  (0.000%)
    country                    null=       0  empty_str=       0  (0.000%)

  S2:

<a id='2'></a>
## 2. Row-Position / ID Leakage Detection

This data looks synthetically generated. Synthetic dedup benchmarks commonly leak the answer 
through row order or ID structure (e.g., row _i_ of S1 maps to row _i_ of S2/S3).

We check:
1. **Numeric ID correlation:** Extract the numeric part of entity_ids and check if matched pairs share similar numeric IDs.
2. **Row-position correlation:** Do matched records tend to appear at the same row index across files?
3. **ID-difference distribution:** For matched pairs, what's the distribution of |id_numeric(S1) - id_numeric(S2/S3)|?

In [ ]:
print('=' * 70)
print('ROW-POSITION / ID LEAKAGE DETECTION')
print('=' * 70)

# Extract numeric portion of entity_id
def extract_numeric_id(eid):
    """Extract numeric part from 'S1-12345' -> 12345"""
    parts = str(eid).split('-', 1)
    if len(parts) == 2:
        try:
            return int(parts[1])
        except ValueError:
            return None
    return None

# Build row-position lookup for each source
s1_rowpos = {eid: i for i, eid in enumerate(s1['entity_id'])}
s2_rowpos = {eid: i for i, eid in enumerate(s2['entity_id'])}
s3_rowpos = {eid: i for i, eid in enumerate(s3['entity_id'])}

# Build numeric-ID lookup
s1_numid = {eid: extract_numeric_id(eid) for eid in s1['entity_id']}
s2_numid = {eid: extract_numeric_id(eid) for eid in s2['entity_id']}
s3_numid = {eid: extract_numeric_id(eid) for eid in s3['entity_id']}

# Descriptive stats on numeric IDs
s1_nums = [v for v in s1_numid.values() if v is not None]
s2_nums = [v for v in s2_numid.values() if v is not None]
s3_nums = [v for v in s3_numid.values() if v is not None]

print('\n--- Numeric ID ranges ---')
for name, nums in [('S1', s1_nums), ('S2', s2_nums), ('S3', s3_nums)]:
    arr = np.array(nums)
    print(f'  {name}: min={arr.min():>12,}  max={arr.max():>12,}  mean={arr.mean():>14.1f}  std={arr.std():>14.1f}')

ROW-POSITION / ID LEAKAGE DETECTION


In [ ]:
# Analyze matched pairs for positional / ID correlation
print('\nAnalyzing ID and position correlation for matched pairs...')

pos_diffs_s2 = []   # row_pos(s1) - row_pos(matched_s2)
pos_diffs_s3 = []
id_diffs_s2 = []    # numeric_id(s1) - numeric_id(matched_s2)
id_diffs_s3 = []
exact_pos_match_s2 = 0
exact_pos_match_s3 = 0
exact_id_match = 0
total_s2_matches = 0
total_s3_matches = 0

for _, row in gt.iterrows():
    s1_eid = row['source1_entity_id']
    mids = row.get('matched_entity_ids', '')
    if pd.isna(mids) or str(mids).strip() == '':
        continue
    
    s1_pos = s1_rowpos.get(s1_eid)
    s1_num = s1_numid.get(s1_eid)
    if s1_pos is None or s1_num is None:
        continue
    
    for mid in str(mids).split(','):
        mid = mid.strip()
        if mid.startswith('S2-'):
            total_s2_matches += 1
            m_pos = s2_rowpos.get(mid)
            m_num = s2_numid.get(mid)
            if m_pos is not None:
                pos_diffs_s2.append(s1_pos - m_pos)
                if s1_pos == m_pos:
                    exact_pos_match_s2 += 1
            if m_num is not None and s1_num is not None:
                id_diffs_s2.append(s1_num - m_num)
                if s1_num == m_num:
                    exact_id_match += 1
        elif mid.startswith('S3-'):
            total_s3_matches += 1
            m_pos = s3_rowpos.get(mid)
            m_num = s3_numid.get(mid)
            if m_pos is not None:
                pos_diffs_s3.append(s1_pos - m_pos)
                if s1_pos == m_pos:
                    exact_pos_match_s3 += 1
            if m_num is not None and s1_num is not None:
                id_diffs_s3.append(s1_num - m_num)
                if s1_num == m_num:
                    exact_id_match += 1

print(f'\n--- Row-position correlation ---')
print(f'  Total S2 matches analyzed: {total_s2_matches:,}')
print(f'  Total S3 matches analyzed: {total_s3_matches:,}')
print(f'  Exact row-position matches (S1↔S2): {exact_pos_match_s2:,} / {total_s2_matches:,} ({exact_pos_match_s2/max(total_s2_matches,1)*100:.4f}%)')
print(f'  Exact row-position matches (S1↔S3): {exact_pos_match_s3:,} / {total_s3_matches:,} ({exact_pos_match_s3/max(total_s3_matches,1)*100:.4f}%)')

# Position difference stats
for name, diffs in [('S1↔S2 position diffs', pos_diffs_s2), ('S1↔S3 position diffs', pos_diffs_s3)]:
    if diffs:
        arr = np.array(diffs)
        print(f'\n  {name}:')
        print(f'    mean={arr.mean():>12.1f}  median={np.median(arr):>12.1f}  std={arr.std():>12.1f}')
        print(f'    |diff|<=10: {np.sum(np.abs(arr)<=10):,} ({np.mean(np.abs(arr)<=10)*100:.4f}%)')
        print(f'    |diff|<=100: {np.sum(np.abs(arr)<=100):,} ({np.mean(np.abs(arr)<=100)*100:.4f}%)')
        print(f'    |diff|<=1000: {np.sum(np.abs(arr)<=1000):,} ({np.mean(np.abs(arr)<=1000)*100:.4f}%)')

In [ ]:
# Numeric ID difference stats
print('--- Numeric ID correlation ---')
print(f'  Exact numeric ID matches across sources: {exact_id_match:,}')

for name, diffs in [('S1↔S2 numeric ID diffs', id_diffs_s2), ('S1↔S3 numeric ID diffs', id_diffs_s3)]:
    if diffs:
        arr = np.array(diffs, dtype=np.float64)
        print(f'\n  {name}:')
        print(f'    mean={arr.mean():>14.1f}  median={np.median(arr):>14.1f}  std={arr.std():>14.1f}')
        print(f'    |diff|==0: {np.sum(arr==0):,} ({np.mean(arr==0)*100:.4f}%)')
        print(f'    |diff|<=100: {np.sum(np.abs(arr)<=100):,} ({np.mean(np.abs(arr)<=100)*100:.4f}%)')
        print(f'    |diff|<=10000: {np.sum(np.abs(arr)<=10000):,} ({np.mean(np.abs(arr)<=10000)*100:.4f}%)')

# Spearman rank correlation between S1 row positions and matched S2/S3 row positions
from scipy.stats import spearmanr, pearsonr
try:
    from scipy.stats import spearmanr, pearsonr
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False

if HAS_SCIPY:
    # Collect (s1_pos, matched_pos) pairs for correlation
    s1_positions_for_s2, s2_positions = [], []
    s1_positions_for_s3, s3_positions = [], []
    
    for _, row in gt.iterrows():
        s1_eid = row['source1_entity_id']
        mids = row.get('matched_entity_ids', '')
        if pd.isna(mids) or str(mids).strip() == '':
            continue
        s1_pos = s1_rowpos.get(s1_eid)
        if s1_pos is None:
            continue
        for mid in str(mids).split(','):
            mid = mid.strip()
            if mid.startswith('S2-') and mid in s2_rowpos:
                s1_positions_for_s2.append(s1_pos)
                s2_positions.append(s2_rowpos[mid])
            elif mid.startswith('S3-') and mid in s3_rowpos:
                s1_positions_for_s3.append(s1_pos)
                s3_positions.append(s3_rowpos[mid])
    
    # Subsample for correlation computation (spearmanr on millions is slow)
    MAX_CORR_SAMPLE = 500_000
    rng = np.random.default_rng(42)
    
    for tag, x_arr, y_arr in [
        ('S1↔S2 row-position', s1_positions_for_s2, s2_positions),
        ('S1↔S3 row-position', s1_positions_for_s3, s3_positions)
    ]:
        n = len(x_arr)
        if n > MAX_CORR_SAMPLE:
            idx = rng.choice(n, MAX_CORR_SAMPLE, replace=False)
            x = np.array(x_arr)[idx]
            y = np.array(y_arr)[idx]
        else:
            x, y = np.array(x_arr), np.array(y_arr)
        
        if len(x) > 1:
            pr, p_p = pearsonr(x, y)
            sr, s_p = spearmanr(x, y)
            print(f'\n  {tag} (n={n:,}, sample={len(x):,}):')
            print(f'    Pearson  r = {pr:+.6f}  (p={p_p:.2e})')
            print(f'    Spearman ρ = {sr:+.6f}  (p={s_p:.2e})')
else:
    print('\n  [scipy not available — skipping rank correlation test]')

# Also check: do S1 numeric IDs and their matched S2/S3 numeric IDs correlate?
if HAS_SCIPY:
    s1_nids_for_s2, s2_nids = [], []
    s1_nids_for_s3, s3_nids = [], []
    for _, row in gt.iterrows():
        s1_eid = row['source1_entity_id']
        mids = row.get('matched_entity_ids', '')
        if pd.isna(mids) or str(mids).strip() == '':
            continue
        s1_n = s1_numid.get(s1_eid)
        if s1_n is None:
            continue
        for mid in str(mids).split(','):
            mid = mid.strip()
            if mid.startswith('S2-'):
                mn = s2_numid.get(mid)
                if mn is not None:
                    s1_nids_for_s2.append(s1_n)
                    s2_nids.append(mn)
            elif mid.startswith('S3-'):
                mn = s3_numid.get(mid)
                if mn is not None:
                    s1_nids_for_s3.append(s1_n)
                    s3_nids.append(mn)
    
    for tag, x_arr, y_arr in [
        ('S1↔S2 numeric-ID', s1_nids_for_s2, s2_nids),
        ('S1↔S3 numeric-ID', s1_nids_for_s3, s3_nids)
    ]:
        n = len(x_arr)
        if n > MAX_CORR_SAMPLE:
            idx = rng.choice(n, MAX_CORR_SAMPLE, replace=False)
            x = np.array(x_arr, dtype=np.float64)[idx]
            y = np.array(y_arr, dtype=np.float64)[idx]
        else:
            x = np.array(x_arr, dtype=np.float64)
            y = np.array(y_arr, dtype=np.float64)
        
        if len(x) > 1:
            pr, p_p = pearsonr(x, y)
            sr, s_p = spearmanr(x, y)
            print(f'\n  {tag} (n={n:,}, sample={len(x):,}):')
            print(f'    Pearson  r = {pr:+.6f}  (p={p_p:.2e})')
            print(f'    Spearman ρ = {sr:+.6f}  (p={s_p:.2e})')

print('\n--- Leakage Verdict ---')
print('If Pearson/Spearman correlations are near 0 and exact-position matches are at')
print('chance level, there is NO positional or ID leakage. If correlations are high')
print('(|r| > 0.1), the problem is fundamentally different and exploitable.')

<a id='3'></a>
## 3. Label Structure: Singleton Rate & Match-Count Distribution

Critical for baseline setting. Under macro F_0.5, singletons score 1.0 when predicted empty and 0.0 when any match is predicted. High singleton rates mean "predict nothing" is a deceptively strong baseline.

In [ ]:
print('=' * 70)
print('LABEL STRUCTURE: SINGLETON RATE & MATCH-COUNT DISTRIBUTION')
print('=' * 70)

# Parse match counts
def count_matches(mids):
    if pd.isna(mids) or str(mids).strip() == '':
        return 0
    return len([x for x in str(mids).split(',') if x.strip()])

gt['n_matches'] = gt['matched_entity_ids'].apply(count_matches)

# Parse into S2-count and S3-count
def count_by_source(mids):
    if pd.isna(mids) or str(mids).strip() == '':
        return 0, 0
    ids = [x.strip() for x in str(mids).split(',') if x.strip()]
    s2c = sum(1 for x in ids if x.startswith('S2-'))
    s3c = sum(1 for x in ids if x.startswith('S3-'))
    return s2c, s3c

s2s3_counts = gt['matched_entity_ids'].apply(count_by_source)
gt['n_s2_matches'] = s2s3_counts.apply(lambda x: x[0])
gt['n_s3_matches'] = s2s3_counts.apply(lambda x: x[1])

n_total = len(gt)
n_singleton = (gt['n_matches'] == 0).sum()
n_nonsingleton = n_total - n_singleton

print(f'\nTotal S1 entities:    {n_total:>10,}')
print(f'Singletons (0 match): {n_singleton:>10,}  ({n_singleton/n_total*100:.2f}%)')
print(f'Non-singletons:       {n_nonsingleton:>10,}  ({n_nonsingleton/n_total*100:.2f}%)')

print(f'\n--- Match count distribution ---')
match_dist = gt['n_matches'].value_counts().sort_index()
print(f'{"n_matches":>10s} {"count":>10s} {"pct":>8s} {"cum_pct":>8s}')
cum = 0
for n, cnt in match_dist.items():
    cum += cnt
    print(f'{n:>10d} {cnt:>10,} {cnt/n_total*100:>7.2f}% {cum/n_total*100:>7.2f}%')

print(f'\nMatch count stats (non-singletons only):')
ns = gt.loc[gt['n_matches'] > 0, 'n_matches']
print(f'  mean={ns.mean():.2f}  median={ns.median():.1f}  max={ns.max()}')
print(f'  p25={ns.quantile(0.25):.0f}  p75={ns.quantile(0.75):.0f}  p95={ns.quantile(0.95):.0f}  p99={ns.quantile(0.99):.0f}')

In [ ]:
# S2 vs S3 match count distribution
print('\n--- S2 vs S3 match count distribution ---')
print(f'{"n_s2":>6s} {"n_s3":>6s} {"count":>10s} {"pct":>8s}')
cross = gt.groupby(['n_s2_matches', 'n_s3_matches']).size().reset_index(name='count')
cross = cross.sort_values('count', ascending=False)
for _, r in cross.head(20).iterrows():
    print(f'{int(r.n_s2_matches):>6d} {int(r.n_s3_matches):>6d} {int(r["count"]):>10,} {r["count"]/n_total*100:>7.2f}%')
if len(cross) > 20:
    print(f'  ... {len(cross) - 20} more combinations')

# How many S1 entities have ONLY S2 matches, ONLY S3 matches, or both?
only_s2 = ((gt['n_s2_matches'] > 0) & (gt['n_s3_matches'] == 0)).sum()
only_s3 = ((gt['n_s2_matches'] == 0) & (gt['n_s3_matches'] > 0)).sum()
both = ((gt['n_s2_matches'] > 0) & (gt['n_s3_matches'] > 0)).sum()
print(f'\n  Has S2 matches only: {only_s2:>10,}  ({only_s2/n_total*100:.2f}%)')
print(f'  Has S3 matches only: {only_s3:>10,}  ({only_s3/n_total*100:.2f}%)')
print(f'  Has both S2 and S3:  {both:>10,}  ({both/n_total*100:.2f}%)')
print(f'  Singleton (neither): {n_singleton:>10,}  ({n_singleton/n_total*100:.2f}%)')

print('\n--- Baseline: "predict nothing" macro F_0.5 ---')
# Singletons score 1.0, non-singletons score 0.0
baseline_f05 = n_singleton / n_total
print(f'  Score = singleton_rate = {baseline_f05:.4f}')

<a id='4'></a>
## 4. Country-Field Trustworthiness

For every GT match pair, verify the S1 and matched S2/S3 country strings agree exactly.
Also audit country spelling/casing across sources.

In [ ]:
print('=' * 70)
print('COUNTRY-FIELD TRUSTWORTHINESS')
print('=' * 70)

# Country distribution per source
print('\n--- Country distribution per source ---')
for name, df in [('S1', s1), ('S2', s2), ('S3', s3)]:
    print(f'\n  {name} ({len(df):,} rows):')
    vc = df['country'].value_counts(dropna=False)
    for country, cnt in vc.items():
        print(f'    {str(country):30s}  {cnt:>10,}  ({cnt/len(df)*100:.2f}%)')

# Country spelling/casing variants
print('\n--- Distinct country strings across all sources ---')
all_countries = set(s1['country'].dropna().unique()) | set(s2['country'].dropna().unique()) | set(s3['country'].dropna().unique())
for c in sorted(all_countries):
    in_s1 = c in s1['country'].values
    in_s2 = c in s2['country'].values
    in_s3 = c in s3['country'].values
    print(f'  "{c}"  S1={in_s1}  S2={in_s2}  S3={in_s3}')

In [ ]:
# Country agreement for matched pairs
print('\n--- Country agreement for GT matched pairs ---')

# Build country lookups
s1_country = dict(zip(s1['entity_id'], s1['country']))
s2_country = dict(zip(s2['entity_id'], s2['country']))
s3_country = dict(zip(s3['entity_id'], s3['country']))

agree_s2 = 0
disagree_s2 = 0
agree_s3 = 0
disagree_s3 = 0
disagree_examples = []

for _, row in gt.iterrows():
    s1_eid = row['source1_entity_id']
    mids = row.get('matched_entity_ids', '')
    if pd.isna(mids) or str(mids).strip() == '':
        continue
    
    s1_c = s1_country.get(s1_eid, None)
    
    for mid in str(mids).split(','):
        mid = mid.strip()
        if mid.startswith('S2-'):
            m_c = s2_country.get(mid, None)
            if s1_c == m_c:
                agree_s2 += 1
            else:
                disagree_s2 += 1
                if len(disagree_examples) < 20:
                    disagree_examples.append((s1_eid, mid, s1_c, m_c))
        elif mid.startswith('S3-'):
            m_c = s3_country.get(mid, None)
            if s1_c == m_c:
                agree_s3 += 1
            else:
                disagree_s3 += 1
                if len(disagree_examples) < 20:
                    disagree_examples.append((s1_eid, mid, s1_c, m_c))

total_s2_pairs = agree_s2 + disagree_s2
total_s3_pairs = agree_s3 + disagree_s3

print(f'  S1↔S2 pairs: {total_s2_pairs:,}')
print(f'    Agree:    {agree_s2:>10,}  ({agree_s2/max(total_s2_pairs,1)*100:.4f}%)')
print(f'    Disagree: {disagree_s2:>10,}  ({disagree_s2/max(total_s2_pairs,1)*100:.4f}%)')
print(f'\n  S1↔S3 pairs: {total_s3_pairs:,}')
print(f'    Agree:    {agree_s3:>10,}  ({agree_s3/max(total_s3_pairs,1)*100:.4f}%)')
print(f'    Disagree: {disagree_s3:>10,}  ({disagree_s3/max(total_s3_pairs,1)*100:.4f}%)')

if disagree_examples:
    print(f'\n  Country disagreement examples:')
    for s1e, me, s1c, mc in disagree_examples[:10]:
        print(f'    {s1e} ({s1c}) ↔ {me} ({mc})')

agreement_rate = (agree_s2 + agree_s3) / max(total_s2_pairs + total_s3_pairs, 1)
print(f'\n  Overall country agreement rate: {agreement_rate*100:.4f}%')
print(f'  → Country is a {"HARD" if agreement_rate > 0.999 else "SOFT"} blocking key')

<a id='5'></a>
## 5. Legal-Suffix Inventory per Country

Catalog the full suffix vocabulary per country and measure what fraction of names carry no suffix.

In [ ]:
print('=' * 70)
print('LEGAL-SUFFIX INVENTORY PER COUNTRY')
print('=' * 70)

# Comprehensive legal suffix patterns
# Using word-boundary regex to find these at end of name or as standalone tokens
LEGAL_SUFFIXES = [
    # US
    'llc', 'l.l.c.', 'l.l.c', 'inc', 'inc.', 'incorporated',
    'corp', 'corp.', 'corporation', 'co', 'co.', 'company',
    'ltd', 'ltd.', 'limited', 'llp', 'l.l.p.', 'l.l.p',
    'lp', 'l.p.', 'pllc', 'p.l.l.c.',
    'pc', 'p.c.', 'pa', 'p.a.',
    # India
    'pvt', 'pvt.', 'private', 'ngo',
    'opc',  # One Person Company
    # France
    'sarl', 's.a.r.l.', 'sas', 's.a.s.', 'sasu', 's.a.s.u.',
    'sa', 's.a.', 'eurl', 'e.u.r.l.', 'sci', 's.c.i.',
    'snc', 's.n.c.',
    # General
    'enterprises', 'enterprise', 'group', 'holdings', 'services',
    'solutions', 'technologies', 'tech', 'industries', 'associates',
    'partners', 'consultants', 'international', 'global',
    'foundation', 'trust', 'society',
]

# Build a fast regex that matches any of these at the end of name
suffix_pattern = re.compile(
    r'\b(' + '|'.join(re.escape(s) for s in sorted(LEGAL_SUFFIXES, key=len, reverse=True)) + r')\s*\.?\s*$',
    re.IGNORECASE
)

# Also a broader pattern: suffix anywhere in the last 3 tokens
suffix_token_pattern = re.compile(
    r'\b(' + '|'.join(re.escape(s) for s in sorted(LEGAL_SUFFIXES, key=len, reverse=True)) + r')\b',
    re.IGNORECASE
)

def find_legal_suffixes(name):
    """Return set of legal suffix tokens found in the name."""
    if pd.isna(name):
        return set()
    return set(m.lower() for m in suffix_token_pattern.findall(str(name)))

def has_trailing_suffix(name):
    """Check if name ends with a legal suffix."""
    if pd.isna(name):
        return False
    return bool(suffix_pattern.search(str(name)))

# Analyze per source, per country
for src_name, df in [('S1', s1), ('S2', s2), ('S3', s3)]:
    print(f'\n{"=" * 50}')
    print(f'{src_name}: Legal suffix analysis')
    print(f'{"=" * 50}')
    
    for country in sorted(df['country'].dropna().unique()):
        mask = df['country'] == country
        names = df.loc[mask, 'business_name']
        n_total_c = len(names)
        
        # Count each suffix
        suffix_counts = Counter()
        n_has_suffix = 0
        n_trailing_suffix = 0
        
        for name in names:
            suffixes = find_legal_suffixes(name)
            if suffixes:
                n_has_suffix += 1
                suffix_counts.update(suffixes)
            if has_trailing_suffix(name):
                n_trailing_suffix += 1
        
        n_no_suffix = n_total_c - n_has_suffix
        print(f'\n  Country: {country} ({n_total_c:,} names)')
        print(f'    Has any suffix token:  {n_has_suffix:>10,}  ({n_has_suffix/n_total_c*100:.1f}%)')
        print(f'    Has trailing suffix:   {n_trailing_suffix:>10,}  ({n_trailing_suffix/n_total_c*100:.1f}%)')
        print(f'    No suffix at all:      {n_no_suffix:>10,}  ({n_no_suffix/n_total_c*100:.1f}%)')
        print(f'    Top suffixes:')
        for suf, cnt in suffix_counts.most_common(15):
            print(f'      {suf:20s}  {cnt:>10,}  ({cnt/n_total_c*100:.1f}%)')

In [ ]:
# Suffix consistency across matched pairs
print('\n--- Suffix consistency across matched pairs ---')

# Build name lookups
s1_name = dict(zip(s1['entity_id'], s1['business_name']))
s2_name = dict(zip(s2['entity_id'], s2['business_name']))
s3_name = dict(zip(s3['entity_id'], s3['business_name']))

suffix_agree = 0
suffix_disagree = 0
suffix_disagree_types = Counter()  # (s1_suffix, matched_suffix) pairs
suffix_disagree_examples = []

for _, row in gt.head(200_000).iterrows():  # Sample for speed on this analysis
    s1_eid = row['source1_entity_id']
    mids = row.get('matched_entity_ids', '')
    if pd.isna(mids) or str(mids).strip() == '':
        continue
    
    s1_n = s1_name.get(s1_eid, '')
    s1_suf = frozenset(find_legal_suffixes(s1_n))
    
    for mid in str(mids).split(','):
        mid = mid.strip()
        if mid.startswith('S2-'):
            m_n = s2_name.get(mid, '')
        elif mid.startswith('S3-'):
            m_n = s3_name.get(mid, '')
        else:
            continue
        m_suf = frozenset(find_legal_suffixes(m_n))
        
        if s1_suf == m_suf:
            suffix_agree += 1
        else:
            suffix_disagree += 1
            suffix_disagree_types[(tuple(sorted(s1_suf)), tuple(sorted(m_suf)))] += 1
            if len(suffix_disagree_examples) < 10:
                suffix_disagree_examples.append((s1_n, m_n, s1_suf, m_suf))

total_pairs = suffix_agree + suffix_disagree
print(f'  Pairs analyzed: {total_pairs:,}')
print(f'  Suffix sets agree:    {suffix_agree:>10,}  ({suffix_agree/max(total_pairs,1)*100:.1f}%)')
print(f'  Suffix sets disagree: {suffix_disagree:>10,}  ({suffix_disagree/max(total_pairs,1)*100:.1f}%)')

print(f'\n  Top disagreement patterns:')
for (s1s, ms), cnt in sorted(suffix_disagree_types.items(), key=lambda x: -x[1])[:15]:
    print(f'    S1={set(s1s) or "∅":30s}  matched={set(ms) or "∅":30s}  count={cnt:,}')

if suffix_disagree_examples:
    print(f'\n  Disagreement examples:')
    for s1n, mn, s1s, ms in suffix_disagree_examples[:5]:
        print(f'    S1: "{s1n}" ({s1s})')
        print(f'    M:  "{mn}" ({ms})')
        print()

<a id='6'></a>
## 6. Address Component Order & Postal-Code Presence

Check whether address ordering is genuinely unfixed and whether postal codes are rare/absent.

In [ ]:
print('=' * 70)
print('ADDRESS COMPONENT ORDER & POSTAL-CODE PRESENCE')
print('=' * 70)

# --- Postal code detection ---
# US ZIP: 5 digits or 5+4
# India PIN: 6 digits
# France postal: 5 digits  
# Generic: look for standalone digit sequences of 5-6 digits

zip_us = re.compile(r'\b\d{5}(-\d{4})?\b')
pin_india = re.compile(r'\b\d{6}\b')
postal_france = re.compile(r'\b\d{5}\b')
any_postal = re.compile(r'\b\d{5,6}\b')

# US state abbreviations for address order detection
US_STATES_ABBR = {
    'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID','IL','IN',
    'IA','KS','KY','LA','ME','MD','MA','MI','MN','MS','MO','MT','NE','NV',
    'NH','NJ','NM','NY','NC','ND','OH','OK','OR','PA','RI','SC','SD','TN',
    'TX','UT','VT','VA','WA','WV','WI','WY','DC'
}

US_STATES_FULL = {
    'alabama','alaska','arizona','arkansas','california','colorado','connecticut',
    'delaware','florida','georgia','hawaii','idaho','illinois','indiana','iowa',
    'kansas','kentucky','louisiana','maine','maryland','massachusetts','michigan',
    'minnesota','mississippi','missouri','montana','nebraska','nevada',
    'new hampshire','new jersey','new mexico','new york','north carolina',
    'north dakota','ohio','oklahoma','oregon','pennsylvania','rhode island',
    'south carolina','south dakota','tennessee','texas','utah','vermont',
    'virginia','washington','west virginia','wisconsin','wyoming',
    'district of columbia'
}

INDIA_STATES = {
    'andhra pradesh', 'arunachal pradesh', 'assam', 'bihar', 'chhattisgarh',
    'goa', 'gujarat', 'haryana', 'himachal pradesh', 'jharkhand', 'karnataka',
    'kerala', 'madhya pradesh', 'maharashtra', 'manipur', 'meghalaya', 'mizoram',
    'nagaland', 'odisha', 'punjab', 'rajasthan', 'sikkim', 'tamil nadu',
    'telangana', 'tripura', 'uttar pradesh', 'uttarakhand', 'west bengal',
    'delhi', 'new delhi', 'chandigarh', 'puducherry', 'jammu and kashmir',
    'ladakh', 'andaman and nicobar', 'dadra and nagar haveli', 'daman and diu',
    'lakshadweep'
}

def detect_address_order_us(addr):
    """Detect if US address starts with street or state.
    Returns: 'street_first', 'state_first', 'ambiguous', or 'no_state'"""
    if pd.isna(addr) or not str(addr).strip():
        return 'empty'
    addr = str(addr).strip()
    parts = [p.strip() for p in addr.split(',')]
    if len(parts) < 2:
        return 'no_comma'
    
    first = parts[0].strip().upper()
    last = parts[-1].strip().upper()
    
    # Check if first part is a state abbreviation
    first_tokens = first.split()
    last_tokens = last.split()
    
    first_is_state = (first in US_STATES_ABBR or 
                      first.lower() in US_STATES_FULL or
                      (len(first_tokens) > 0 and first_tokens[0] in US_STATES_ABBR))
    
    last_is_state = (last in US_STATES_ABBR or 
                     last.lower() in US_STATES_FULL or
                     (len(last_tokens) > 0 and last_tokens[-1] in US_STATES_ABBR))
    
    # Check if first part looks like a street (starts with a number)
    first_is_street = bool(re.match(r'^\d+', first))
    
    if first_is_state and not first_is_street:
        return 'state_first'
    elif last_is_state:
        return 'street_first'
    elif first_is_street:
        return 'street_first'
    else:
        return 'ambiguous'

# Analyze address patterns per source per country
for src_name, df in [('S1', s1), ('S2', s2), ('S3', s3)]:
    print(f'\n--- {src_name}: Postal code presence ---')
    for country in sorted(df['country'].dropna().unique()):
        mask = df['country'] == country
        addrs = df.loc[mask, 'business_address'].fillna('')
        n_total_c = len(addrs)
        n_has_postal = addrs.apply(lambda x: bool(any_postal.search(str(x)))).sum()
        n_empty = (addrs == '').sum()
        print(f'  {country:15s}: {n_has_postal:>8,}/{n_total_c:>8,} ({n_has_postal/max(n_total_c,1)*100:.1f}%) have postal code pattern, {n_empty:,} empty')

In [ ]:
# Address order analysis for US addresses in each source
print('\n--- Address component order (US addresses) ---')
for src_name, df in [('S1', s1), ('S2', s2), ('S3', s3)]:
    us_addrs = df.loc[df['country'] == 'US', 'business_address']
    order_counts = Counter()
    for addr in us_addrs:
        order_counts[detect_address_order_us(addr)] += 1
    
    n_us = len(us_addrs)
    print(f'\n  {src_name} US addresses ({n_us:,}):')
    for order, cnt in order_counts.most_common():
        print(f'    {order:20s}  {cnt:>10,}  ({cnt/max(n_us,1)*100:.1f}%)')

# India address analysis
print('\n--- Address component order (India addresses) ---')
for src_name, df in [('S1', s1), ('S2', s2), ('S3', s3)]:
    india_addrs = df.loc[df['country'] == 'India', 'business_address'].fillna('')
    n_india = len(india_addrs)
    
    # Check if state appears at end (typical) or beginning
    state_last = 0
    state_first = 0
    has_state = 0
    
    for addr in india_addrs:
        addr_lower = str(addr).lower().strip()
        parts = [p.strip().lower() for p in addr_lower.split(',')]
        if len(parts) < 2:
            continue
        found_state_pos = None
        for i, p in enumerate(parts):
            if p in INDIA_STATES:
                found_state_pos = i
                break
        if found_state_pos is not None:
            has_state += 1
            if found_state_pos == len(parts) - 1:
                state_last += 1
            elif found_state_pos == 0:
                state_first += 1
    
    print(f'\n  {src_name} India addresses ({n_india:,}):')
    print(f'    Has recognizable state: {has_state:>10,} ({has_state/max(n_india,1)*100:.1f}%)')
    print(f'    State at end:           {state_last:>10,} ({state_last/max(n_india,1)*100:.1f}%)')
    print(f'    State at beginning:     {state_first:>10,} ({state_first/max(n_india,1)*100:.1f}%)')

In [ ]:
# Address field: length distribution, comma count, component count
print('\n--- Address structure stats ---')
for src_name, df in [('S1', s1), ('S2', s2), ('S3', s3)]:
    print(f'\n  {src_name}:')
    for country in sorted(df['country'].dropna().unique()):
        addrs = df.loc[df['country'] == country, 'business_address'].fillna('')
        n = len(addrs)
        lengths = addrs.str.len()
        n_commas = addrs.str.count(',')
        n_empty = (addrs == '').sum()
        
        print(f'    {country:15s} (n={n:,}):')
        print(f'      Length: mean={lengths.mean():.0f}  median={lengths.median():.0f}  p95={lengths.quantile(0.95):.0f}  max={lengths.max()}')
        print(f'      Commas: mean={n_commas.mean():.1f}  median={n_commas.median():.0f}  max={n_commas.max()}')
        print(f'      Empty:  {n_empty:,} ({n_empty/n*100:.1f}%)')

<a id='7'></a>
## 7. Junk Characters & Encoding Artifacts

Scan all three sources for non-alphanumeric junk, leading/trailing symbols, mojibake, or control characters.

In [ ]:
print('=' * 70)
print('JUNK CHARACTERS & ENCODING ARTIFACTS')
print('=' * 70)

def analyze_junk(series, field_name, src_name):
    """Analyze junk characters in a text series."""
    results = {}
    n = len(series)
    texts = series.fillna('').astype(str)
    
    # 1. Leading/trailing non-alphanumeric (excluding quotes, parens, common punct)
    leading_junk = texts.str.match(r'^[^a-zA-Z0-9\s"\'\'\(]')
    trailing_junk = texts.str.match(r'.*[^a-zA-Z0-9\s\."\'\'\)]$')
    
    # 2. Control characters (outside normal printable range, but allow Unicode letters)
    control_chars = texts.apply(lambda x: bool(re.search(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', x)))
    
    # 3. Consecutive special characters (<<, >>, --, etc.)
    consec_special = texts.str.contains(r'[<>]{2,}|[-]{3,}|[=]{2,}|[#]{2,}|[*]{2,}', regex=True, na=False)
    
    # 4. Leading -- or << or >> or other junk prefixes
    leading_symbols = texts.str.match(r'^\s*[\-<>=#*@!~`]{2,}')
    
    # 5. Non-Latin, non-Devanagari/other-Indic scripts (potential mojibake)
    # Check for characters in the Latin Extended / Combining range that might be mojibake
    mojibake_pattern = re.compile(r'[\xc0-\xff]{3,}')  # Consecutive high-byte Latin chars
    possible_mojibake = texts.apply(lambda x: bool(mojibake_pattern.search(x)))
    
    # 6. Mixed scripts (Latin + Devanagari in same field)
    has_devanagari = texts.str.contains(r'[\u0900-\u097F]', regex=True, na=False)
    has_latin = texts.str.contains(r'[a-zA-Z]', regex=True, na=False)
    mixed_script = has_devanagari & has_latin
    
    # 7. Tabs or multiple spaces
    has_tabs = texts.str.contains('\t', regex=False, na=False)
    multi_spaces = texts.str.contains(r'  +', regex=True, na=False)
    
    print(f'\n  {src_name} → {field_name} ({n:,} values):')
    for label, mask in [
        ('Leading non-alnum', leading_junk),
        ('Trailing non-alnum', trailing_junk),
        ('Control characters', control_chars),
        ('Consecutive specials (<<, --, etc)', consec_special),
        ('Leading symbol runs (--X, <<X)', leading_symbols),
        ('Possible mojibake', possible_mojibake),
        ('Mixed Latin+Devanagari', mixed_script),
        ('Contains tabs', has_tabs),
        ('Multiple consecutive spaces', multi_spaces),
    ]:
        cnt = mask.sum()
        print(f'    {label:45s}  {cnt:>8,}  ({cnt/n*100:.3f}%)')
    
    return texts

for src_name, df in [('S1', s1), ('S2', s2), ('S3', s3)]:
    for field in ['business_name', 'business_address']:
        analyze_junk(df[field], field, src_name)

In [ ]:
# Catalog the specific non-standard characters found
print('\n--- Non-standard character inventory ---')

def catalog_special_chars(series, field_name, src_name):
    """Find and count every non-ASCII, non-standard character."""
    char_counts = Counter()
    leading_char_counts = Counter()
    
    for text in series.fillna('').astype(str):
        # Non-ASCII characters
        for ch in text:
            if ord(ch) > 127 or (ord(ch) < 32 and ch not in '\n\r\t'):
                cat = unicodedata.category(ch)
                name = unicodedata.name(ch, f'U+{ord(ch):04X}')
                char_counts[(ch, name, cat)] += 1
        
        # Leading non-alnum characters
        stripped = text.lstrip()
        if stripped and not stripped[0].isalnum() and stripped[0] not in '("\'':
            leading_char_counts[stripped[:3]] += 1
    
    print(f'\n  {src_name} → {field_name}:')
    print(f'    Top non-ASCII characters:')
    for (ch, name, cat), cnt in sorted(char_counts.items(), key=lambda x: -x[1])[:25]:
        print(f'      "{ch}"  U+{ord(ch):04X}  {name:40s}  cat={cat}  count={cnt:,}')
    
    if leading_char_counts:
        print(f'    Leading non-alnum prefixes (top 15):')
        for prefix, cnt in leading_char_counts.most_common(15):
            print(f'      "{prefix}"  count={cnt:,}')

for src_name, df in [('S1', s1), ('S2', s2), ('S3', s3)]:
    catalog_special_chars(df['business_name'], 'business_name', src_name)

In [ ]:
# Show examples of junk / leading symbols in business names
print('\n--- Example business names with leading junk ---')
for src_name, df in [('S1', s1), ('S2', s2), ('S3', s3)]:
    junk_mask = df['business_name'].fillna('').str.match(r'^\s*[\-<>=#*@!~`\.]{1,}')
    junk_names = df.loc[junk_mask, ['entity_id', 'business_name', 'country']]
    n_junk = len(junk_names)
    print(f'\n  {src_name}: {n_junk:,} names with leading junk symbols')
    if n_junk > 0:
        print(junk_names.head(10).to_string())

<a id='8'></a>
## 8. Reverse-Engineering the Corruption Function

Take real GT pairs and diff each S1 field against its matched S2/S3 field — token by token, character by character — to catalog the specific transformations used.

In [ ]:
print('=' * 70)
print('REVERSE-ENGINEERING THE CORRUPTION FUNCTION')
print('=' * 70)

# Build fast lookups
# Build fast O(1) address lookups (s1_name already exists from Section 4)
s1_addr = dict(zip(s1['entity_id'], s1['business_address']))
s2_addr = dict(zip(s2['entity_id'], s2['business_address']))
s3_addr = dict(zip(s3['entity_id'], s3['business_address']))

# We'll analyze a substantial sample (first 100k GT rows with matches)
CORRUPTION_SAMPLE = 100_000

# Track transformation types
name_transforms = Counter()  # high-level categories
addr_transforms = Counter()
token_additions = Counter()  # tokens added in S2/S3 not in S1
token_removals = Counter()   # tokens in S1 not in S2/S3
abbreviation_pairs = Counter()  # (long_form, short_form) or vice versa

name_edit_distances = []
addr_edit_distances = []
name_token_jaccard = []
addr_token_jaccard = []

# Detailed character-level analysis
char_subs = Counter()  # (original_char, corrupted_char) pairs

# Track specific corruption categories
n_name_identical = 0
n_name_case_only = 0
n_name_suffix_change = 0
n_name_token_reorder = 0
n_name_token_drop = 0
n_name_token_add = 0
n_name_typo = 0
n_name_abbreviation = 0
n_name_script_change = 0  # Latin to Devanagari etc.

n_addr_identical = 0
n_addr_component_reorder = 0
n_addr_abbreviation = 0
n_addr_component_drop = 0

pair_count = 0

def tokenize(text):
    """Simple whitespace + punctuation tokenization."""
    if pd.isna(text):
        return []
    return re.findall(r'[\w]+', str(text).lower())

def is_script_change(t1, t2):
    """Check if one text is Latin and the other is non-Latin (e.g., Devanagari)."""
    latin1 = bool(re.search(r'[a-zA-Z]', str(t1)))
    latin2 = bool(re.search(r'[a-zA-Z]', str(t2)))
    nonlatin1 = bool(re.search(r'[^\x00-\x7F]', str(t1)))
    nonlatin2 = bool(re.search(r'[^\x00-\x7F]', str(t2)))
    return (latin1 and not latin2 and nonlatin2) or (latin2 and not latin1 and nonlatin1)

# Known abbreviation mappings
KNOWN_ABBREVS = {
    'road': 'rd', 'street': 'st', 'avenue': 'ave', 'drive': 'dr',
    'boulevard': 'blvd', 'lane': 'ln', 'court': 'ct', 'place': 'pl',
    'highway': 'hwy', 'apartment': 'apt', 'suite': 'ste', 'building': 'bldg',
    'floor': 'fl', 'mount': 'mt', 'mountain': 'mtn', 'saint': 'st',
    'fort': 'ft', 'north': 'n', 'south': 's', 'east': 'e', 'west': 'w',
    'northeast': 'ne', 'northwest': 'nw', 'southeast': 'se', 'southwest': 'sw',
    'corporation': 'corp', 'incorporated': 'inc', 'limited': 'ltd',
    'private': 'pvt', 'company': 'co',
}
KNOWN_ABBREVS_REV = {v: k for k, v in KNOWN_ABBREVS.items()}

sample_pairs = []

for _, row in gt.iterrows():
    if pair_count >= CORRUPTION_SAMPLE:
        break
    
    s1_eid = row['source1_entity_id']
    mids = row.get('matched_entity_ids', '')
    if pd.isna(mids) or str(mids).strip() == '':
        continue
    
    s1_nm = s1_name.get(s1_eid, '')
    s1_ad_raw = s1_addr.get(s1_eid, '')
    s1_ad = str(s1_ad_raw) if not pd.isna(s1_ad_raw) else ''
    
    for mid in str(mids).split(','):
        mid = mid.strip()
        if pair_count >= CORRUPTION_SAMPLE:
            break
        
        if mid.startswith('S2-'):
            m_nm = s2_name.get(mid, '')
            m_ad = s2_addr.get(mid, '')
        elif mid.startswith('S3-'):
            m_nm = s3_name.get(mid, '')
            m_ad = s3_addr.get(mid, '')
        else:
            continue
        
        if pd.isna(m_nm): m_nm = ''
        if pd.isna(m_ad): m_ad = ''
        if pd.isna(s1_nm): s1_nm = ''
        
        pair_count += 1
        
        # --- Name analysis ---
        s1_nm_str = str(s1_nm)
        m_nm_str = str(m_nm)
        
        if s1_nm_str == m_nm_str:
            n_name_identical += 1
        elif s1_nm_str.lower() == m_nm_str.lower():
            n_name_case_only += 1
        else:
            # Script change?
            if is_script_change(s1_nm_str, m_nm_str):
                n_name_script_change += 1
            else:
                s1_toks = set(tokenize(s1_nm_str))
                m_toks = set(tokenize(m_nm_str))
                
                # Token-level analysis
                added = m_toks - s1_toks
                removed = s1_toks - m_toks
                shared = s1_toks & m_toks
                
                # Check for suffix change
                s1_suf = find_legal_suffixes(s1_nm_str)
                m_suf = find_legal_suffixes(m_nm_str)
                if s1_suf != m_suf:
                    n_name_suffix_change += 1
                
                # Token reorder (same tokens, different order)
                s1_tok_list = tokenize(s1_nm_str)
                m_tok_list = tokenize(m_nm_str)
                if sorted(s1_tok_list) == sorted(m_tok_list) and s1_tok_list != m_tok_list:
                    n_name_token_reorder += 1
                
                if removed - find_legal_suffixes(s1_nm_str):
                    n_name_token_drop += 1
                if added - find_legal_suffixes(m_nm_str):
                    n_name_token_add += 1
                
                # Abbreviation detection
                for r_tok in removed:
                    if r_tok in KNOWN_ABBREVS:
                        if KNOWN_ABBREVS[r_tok] in added:
                            n_name_abbreviation += 1
                            abbreviation_pairs[(r_tok, KNOWN_ABBREVS[r_tok])] += 1
                    elif r_tok in KNOWN_ABBREVS_REV:
                        if KNOWN_ABBREVS_REV[r_tok] in added:
                            n_name_abbreviation += 1
                            abbreviation_pairs[(KNOWN_ABBREVS_REV[r_tok], r_tok)] += 1
                
                # Edit distance (only for same-script pairs)
                if HAS_RAPIDFUZZ:
                    ed = lev.distance(s1_nm_str.lower(), m_nm_str.lower())
                    name_edit_distances.append(ed)
                
                # Token Jaccard
                if s1_toks or m_toks:
                    jacc = len(shared) / len(s1_toks | m_toks) if (s1_toks | m_toks) else 1.0
                    name_token_jaccard.append(jacc)
                
                for tok in added:
                    token_additions[tok] += 1
                for tok in removed:
                    token_removals[tok] += 1
        
        # --- Address analysis ---
        s1_ad_str = str(s1_ad)
        m_ad_str = str(m_ad)
        
        if s1_ad_str == m_ad_str:
            n_addr_identical += 1
        else:
            s1_addr_toks = set(tokenize(s1_ad_str))
            m_addr_toks = set(tokenize(m_ad_str))
            shared_a = s1_addr_toks & m_addr_toks
            
            if s1_addr_toks or m_addr_toks:
                jacc_a = len(shared_a) / len(s1_addr_toks | m_addr_toks)
                addr_token_jaccard.append(jacc_a)
            
            if HAS_RAPIDFUZZ:
                ed_a = lev.distance(s1_ad_str.lower(), m_ad_str.lower())
                addr_edit_distances.append(ed_a)
            
            # Component reorder check
            s1_parts = sorted(p.strip().lower() for p in s1_ad_str.split(','))
            m_parts = sorted(p.strip().lower() for p in m_ad_str.split(','))
            if s1_parts == m_parts and s1_ad_str.lower() != m_ad_str.lower():
                n_addr_component_reorder += 1
        
        # Save some examples for display
        if len(sample_pairs) < 50 and s1_nm_str != m_nm_str:
            sample_pairs.append({
                's1_id': s1_eid, 'matched_id': mid,
                's1_name': s1_nm_str, 'm_name': m_nm_str,
                's1_addr': s1_ad_str, 'm_addr': m_ad_str,
            })

print(f'\nPairs analyzed: {pair_count:,}')

In [ ]:
# Report corruption analysis results
print(f'\n--- Name corruption categories (out of {pair_count:,} pairs) ---')
for label, cnt in [
    ('Identical', n_name_identical),
    ('Case-only difference', n_name_case_only),
    ('Script change (e.g. Latin→Devanagari)', n_name_script_change),
    ('Suffix added/removed/changed', n_name_suffix_change),
    ('Token reorder (same tokens, diff order)', n_name_token_reorder),
    ('Token dropped', n_name_token_drop),
    ('Token added', n_name_token_add),
    ('Known abbreviation swap', n_name_abbreviation),
]:
    print(f'  {label:50s}  {cnt:>8,}  ({cnt/max(pair_count,1)*100:.1f}%)')

print(f'\n--- Address corruption categories ---')
for label, cnt in [
    ('Identical', n_addr_identical),
    ('Component reorder', n_addr_component_reorder),
]:
    print(f'  {label:50s}  {cnt:>8,}  ({cnt/max(pair_count,1)*100:.1f}%)')

# Edit distance distribution
if name_edit_distances:
    ned = np.array(name_edit_distances)
    print(f'\n--- Name edit distance (non-identical, same-script pairs) ---')
    print(f'  n={len(ned):,}  mean={ned.mean():.1f}  median={np.median(ned):.0f}  p25={np.percentile(ned,25):.0f}  p75={np.percentile(ned,75):.0f}  p95={np.percentile(ned,95):.0f}  max={ned.max()}')
    # Histogram buckets
    for lo, hi in [(0,0),(1,2),(3,5),(6,10),(11,20),(21,50),(51,100),(101,999)]:
        c = np.sum((ned >= lo) & (ned <= hi))
        print(f'    [{lo:>3d}-{hi:>3d}]: {c:>8,} ({c/len(ned)*100:.1f}%)')

if addr_edit_distances:
    aed = np.array(addr_edit_distances)
    print(f'\n--- Address edit distance (non-identical pairs) ---')
    print(f'  n={len(aed):,}  mean={aed.mean():.1f}  median={np.median(aed):.0f}  p25={np.percentile(aed,25):.0f}  p75={np.percentile(aed,75):.0f}  p95={np.percentile(aed,95):.0f}  max={aed.max()}')

if name_token_jaccard:
    ntj = np.array(name_token_jaccard)
    print(f'\n--- Name token Jaccard (non-identical, same-script pairs) ---')
    print(f'  n={len(ntj):,}  mean={ntj.mean():.3f}  median={np.median(ntj):.3f}  p5={np.percentile(ntj,5):.3f}  p25={np.percentile(ntj,25):.3f}')
    for lo_f, hi_f in [(0.0,0.1),(0.1,0.3),(0.3,0.5),(0.5,0.7),(0.7,0.9),(0.9,1.01)]:
        c = np.sum((ntj >= lo_f) & (ntj < hi_f))
        print(f'    [{lo_f:.1f}-{hi_f:.1f}): {c:>8,} ({c/len(ntj)*100:.1f}%)')

In [ ]:
# Most commonly added / removed tokens
print('\n--- Tokens most commonly ADDED in S2/S3 (not in S1 name) ---')
for tok, cnt in token_additions.most_common(30):
    print(f'  {tok:25s}  {cnt:>6,}')

print('\n--- Tokens most commonly REMOVED from S1 name (not in S2/S3) ---')
for tok, cnt in token_removals.most_common(30):
    print(f'  {tok:25s}  {cnt:>6,}')

print('\n--- Most common abbreviation swaps ---')
for (long_f, short_f), cnt in abbreviation_pairs.most_common(20):
    print(f'  {long_f:20s} ↔ {short_f:15s}  {cnt:>6,}')

In [ ]:
# Show example matched pairs with differences highlighted
print('\n--- Example matched pairs (name differences) ---')
for i, p in enumerate(sample_pairs[:20]):
    print(f'\nPair {i+1}: {p["s1_id"]} ↔ {p["matched_id"]}')
    print(f'  S1 name: "{p["s1_name"]}"')
    print(f'  M  name: "{p["m_name"]}"')
    if p['s1_addr'] != p['m_addr']:
        print(f'  S1 addr: "{p["s1_addr"]}"')
        print(f'  M  addr: "{p["m_addr"]}"')

<a id='9'></a>
## 9. Within-Source Name / Address Collision Risk

How often does the same (or normalized-identical) business_name attach to multiple distinct entity_ids within one source? Same for address.

In [ ]:
print('=' * 70)
print('WITHIN-SOURCE NAME / ADDRESS COLLISION RISK')
print('=' * 70)

def normalize_basic(text):
    """Basic normalization: lowercase, strip, collapse whitespace."""
    if pd.isna(text):
        return ''
    t = str(text).lower().strip()
    t = re.sub(r'\s+', ' ', t)
    # Remove common punctuation
    t = re.sub(r'[.,;:!?\'"()\[\]{}]', '', t)
    return t.strip()

for src_name, df in [('S1', s1), ('S2', s2), ('S3', s3)]:
    print(f'\n--- {src_name} ({len(df):,} rows) ---')
    
    # Exact name collisions
    name_groups = df.groupby('business_name')['entity_id'].nunique()
    colliding_names = name_groups[name_groups > 1]
    n_colliding = len(colliding_names)
    n_entities_in_collision = colliding_names.sum()
    print(f'  Exact business_name collisions:')
    print(f'    Distinct names mapping to >1 entity: {n_colliding:,}')
    print(f'    Total entities involved: {n_entities_in_collision:,} ({n_entities_in_collision/len(df)*100:.2f}%)')
    if n_colliding > 0:
        print(f'    Top colliding names:')
        for name_val, cnt in colliding_names.nlargest(10).items():
            display_name = str(name_val)[:80]
            print(f'      "{display_name}" → {cnt} entities')
    
    # Normalized name collisions
    df_temp = df.copy()
    df_temp['_norm_name'] = df_temp['business_name'].apply(normalize_basic)
    norm_groups = df_temp.groupby('_norm_name')['entity_id'].nunique()
    norm_colliding = norm_groups[norm_groups > 1]
    n_norm_colliding = len(norm_colliding)
    n_norm_entities = norm_colliding.sum()
    print(f'\n  Normalized name collisions:')
    print(f'    Distinct norm-names mapping to >1 entity: {n_norm_colliding:,}')
    print(f'    Total entities involved: {n_norm_entities:,} ({n_norm_entities/len(df)*100:.2f}%)')
    
    # Exact address collisions
    addr_groups = df.groupby('business_address')['entity_id'].nunique()
    colliding_addrs = addr_groups[addr_groups > 1]
    n_addr_colliding = len(colliding_addrs)
    n_addr_entities = colliding_addrs.sum()
    print(f'\n  Exact business_address collisions:')
    print(f'    Distinct addresses mapping to >1 entity: {n_addr_colliding:,}')
    print(f'    Total entities involved: {n_addr_entities:,} ({n_addr_entities/len(df)*100:.2f}%)')
    if n_addr_colliding > 0:
        print(f'    Top colliding addresses:')
        for addr_val, cnt in colliding_addrs.nlargest(10).items():
            display_addr = str(addr_val)[:80]
            print(f'      "{display_addr}" → {cnt} entities')
    
    del df_temp

In [ ]:
# Cross-entity collision: how often would name-only or address-only matching cause false merges?
print('\n--- False merge risk from name-only or address-only matching ---')

# For S1 entities specifically (since we match FROM S1)
# How many distinct S1 entity pairs share the same business name?
s1_name_groups = s1.groupby(s1['business_name'].apply(normalize_basic))['entity_id'].apply(list)
multi_name_groups = s1_name_groups[s1_name_groups.apply(len) > 1]

total_false_merge_risk_name = sum(len(g) * (len(g) - 1) // 2 for g in multi_name_groups)
print(f'  S1: {len(multi_name_groups):,} normalized names shared by multiple entities')
print(f'  S1: {total_false_merge_risk_name:,} entity-pairs share same normalized name')

# Same for address
s1_addr_groups = s1.groupby(s1['business_address'].apply(normalize_basic))['entity_id'].apply(list)
multi_addr_groups = s1_addr_groups[s1_addr_groups.apply(len) > 1]
total_false_merge_risk_addr = sum(len(g) * (len(g) - 1) // 2 for g in multi_addr_groups)
print(f'  S1: {len(multi_addr_groups):,} normalized addresses shared by multiple entities')
print(f'  S1: {total_false_merge_risk_addr:,} entity-pairs share same normalized address')

<a id='10'></a>
## 10. Additional Observations

Miscellaneous analyses that further inform the ML pipeline.

In [ ]:
print('=' * 70)
print('ADDITIONAL OBSERVATIONS')
print('=' * 70)

# 10a. Name length distribution per source and country
print('\n--- Business name length (chars) per source × country ---')
for src_name, df in [('S1', s1), ('S2', s2), ('S3', s3)]:
    print(f'\n  {src_name}:')
    for country in sorted(df['country'].dropna().unique()):
        names = df.loc[df['country'] == country, 'business_name'].fillna('').str.len()
        print(f'    {country:15s}: mean={names.mean():.0f}  median={names.median():.0f}  p95={names.quantile(0.95):.0f}  max={names.max()}')

# 10b. Token count distribution  
print('\n--- Business name token count per source × country ---')
for src_name, df in [('S1', s1), ('S2', s2), ('S3', s3)]:
    print(f'\n  {src_name}:')
    for country in sorted(df['country'].dropna().unique()):
        tok_counts = df.loc[df['country'] == country, 'business_name'].fillna('').apply(lambda x: len(x.split()))
        print(f'    {country:15s}: mean={tok_counts.mean():.1f}  median={tok_counts.median():.0f}  p95={tok_counts.quantile(0.95):.0f}  max={tok_counts.max()}')

In [ ]:
# 10c. S2/S3 size ratios relative to S1
print('\n--- Source size ratios ---')
print(f'  S2/S1 ratio: {len(s2)/len(s1):.3f}x')
print(f'  S3/S1 ratio: {len(s3)/len(s1):.3f}x')
print(f'  (S2+S3)/S1 ratio: {(len(s2)+len(s3))/len(s1):.3f}x')

# Country composition per source
print('\n--- Country composition per source ---')
for src_name, df in [('S1', s1), ('S2', s2), ('S3', s3)]:
    vc = df['country'].value_counts()
    total = len(df)
    print(f'\n  {src_name} ({total:,}):')
    for c, n in vc.items():
        print(f'    {c:15s}: {n:>10,} ({n/total*100:.1f}%)')

# 10d. Matches per S2/S3 entity (are S2/S3 entities matched to multiple S1 entities?)
print('\n--- S2/S3 entity reuse (matched to multiple S1 entities) ---')
s2_match_count = Counter()
s3_match_count = Counter()
for _, row in gt.iterrows():
    mids = row.get('matched_entity_ids', '')
    if pd.isna(mids) or str(mids).strip() == '':
        continue
    for mid in str(mids).split(','):
        mid = mid.strip()
        if mid.startswith('S2-'):
            s2_match_count[mid] += 1
        elif mid.startswith('S3-'):
            s3_match_count[mid] += 1

s2_reuse = Counter(s2_match_count.values())
s3_reuse = Counter(s3_match_count.values())

print(f'\n  S2 entities appearing in GT: {len(s2_match_count):,} / {len(s2):,} ({len(s2_match_count)/len(s2)*100:.1f}%)')
print(f'  S2 match-to-S1 count distribution:')
for n_times in sorted(s2_reuse.keys()):
    print(f'    Matched to {n_times} S1 entity(ies): {s2_reuse[n_times]:,}')

print(f'\n  S3 entities appearing in GT: {len(s3_match_count):,} / {len(s3):,} ({len(s3_match_count)/len(s3)*100:.1f}%)')
print(f'  S3 match-to-S1 count distribution:')
for n_times in sorted(s3_reuse.keys()):
    print(f'    Matched to {n_times} S1 entity(ies): {s3_reuse[n_times]:,}')

In [ ]:
# 10e. Devanagari / non-Latin script analysis
print('\n--- Non-Latin script presence ---')
devanagari_re = re.compile(r'[\u0900-\u097F]')  # Devanagari
tamil_re = re.compile(r'[\u0B80-\u0BFF]')       # Tamil
kannada_re = re.compile(r'[\u0C80-\u0CFF]')     # Kannada
bengali_re = re.compile(r'[\u0980-\u09FF]')     # Bengali
telugu_re = re.compile(r'[\u0C00-\u0C7F]')      # Telugu
gujarati_re = re.compile(r'[\u0A80-\u0AFF]')    # Gujarati
arabic_re = re.compile(r'[\u0600-\u06FF]')      # Arabic
cjk_re = re.compile(r'[\u4E00-\u9FFF]')         # CJK

script_patterns = [
    ('Devanagari', devanagari_re),
    ('Tamil', tamil_re),
    ('Kannada', kannada_re),
    ('Bengali', bengali_re),
    ('Telugu', telugu_re),
    ('Gujarati', gujarati_re),
    ('Arabic', arabic_re),
    ('CJK', cjk_re),
]

for src_name, df in [('S1', s1), ('S2', s2), ('S3', s3)]:
    print(f'\n  {src_name} business_name:')
    names = df['business_name'].fillna('')
    for script_name, pattern in script_patterns:
        cnt = names.apply(lambda x: bool(pattern.search(str(x)))).sum()
        if cnt > 0:
            print(f'    {script_name:15s}: {cnt:>8,} ({cnt/len(df)*100:.2f}%)')
    
    print(f'  {src_name} business_address:')
    addrs = df['business_address'].fillna('')
    for script_name, pattern in script_patterns:
        cnt = addrs.apply(lambda x: bool(pattern.search(str(x)))).sum()
        if cnt > 0:
            print(f'    {script_name:15s}: {cnt:>8,} ({cnt/len(df)*100:.2f}%)')

In [ ]:
# 10f. Ampersand vs 'and', punctuation patterns
print('\n--- Ampersand (&) vs "and" in business names ---')
for src_name, df in [('S1', s1), ('S2', s2), ('S3', s3)]:
    names = df['business_name'].fillna('')
    n_amp = names.str.contains('&', regex=False).sum()
    n_and = names.str.contains(r'\band\b', case=False, regex=True).sum()
    n_both = (names.str.contains('&', regex=False) & names.str.contains(r'\band\b', case=False, regex=True)).sum()
    print(f'  {src_name}: & = {n_amp:,}  "and" = {n_and:,}  both = {n_both:,}')

# 10g. .com / website patterns in business names
print('\n--- .com / website patterns in business names ---')
for src_name, df in [('S1', s1), ('S2', s2), ('S3', s3)]:
    names = df['business_name'].fillna('')
    n_dotcom = names.str.contains(r'\.(com|org|net|in|co|io|biz)', case=False, regex=True).sum()
    print(f'  {src_name}: {n_dotcom:,} names contain domain-like patterns ({n_dotcom/len(df)*100:.2f}%)')
    if n_dotcom > 0:
        examples = df.loc[names.str.contains(r'\.(com|org|net|in|co|io|biz)', case=False, regex=True, na=False), 'business_name'].head(5)
        for ex in examples:
            print(f'    "{ex}"')

In [ ]:
# 10h. "Near", "Opp", "Behind" landmark references in addresses (India-specific)
print('\n--- Landmark-based address references ---')
landmark_patterns = [
    ('near', r'\bnear\b'),
    ('opp/opposite', r'\b(opp\.?|opposite)\b'),
    ('behind', r'\bbehind\b'),
    ('beside/adjacent', r'\b(beside|adjacent)\b'),
    ('above', r'\babove\b'),
    ('below', r'\bbelow\b'),
    ('nagar', r'\bnagar\b'),
    ('colony', r'\bcolony\b'),
    ('bazar/bazaar', r'\bbaz[az]ar\b'),
    ('marg/road', r'\b(marg|marg\.?)\b'),
    ('chowk', r'\bchowk\b'),
    ('gali/lane', r'\bgali\b'),
]

for src_name, df in [('S1', s1), ('S2', s2), ('S3', s3)]:
    india_addrs = df.loc[df['country'] == 'India', 'business_address'].fillna('')
    n_india = len(india_addrs)
    if n_india == 0:
        continue
    print(f'\n  {src_name} India addresses ({n_india:,}):')
    for label, pattern in landmark_patterns:
        cnt = india_addrs.str.contains(pattern, case=False, regex=True).sum()
        print(f'    {label:25s}: {cnt:>8,} ({cnt/n_india*100:.1f}%)')

In [ ]:
# 10i. How many S2/S3 entities are NOT matched to any S1 entity (true negatives)?
print('\n--- Unmatched S2/S3 entities (not in any GT pair) ---')
all_matched_s2 = set(s2_match_count.keys())
all_matched_s3 = set(s3_match_count.keys())

unmatched_s2 = len(s2) - len(all_matched_s2)
unmatched_s3 = len(s3) - len(all_matched_s3)

print(f'  S2 total: {len(s2):,}  matched: {len(all_matched_s2):,}  unmatched: {unmatched_s2:,} ({unmatched_s2/len(s2)*100:.1f}%)')
print(f'  S3 total: {len(s3):,}  matched: {len(all_matched_s3):,}  unmatched: {unmatched_s3:,} ({unmatched_s3/len(s3)*100:.1f}%)')
print(f'\n  → {unmatched_s2 + unmatched_s3:,} total distractor entities in S2+S3')
print(f'  → Distractor ratio: {(unmatched_s2+unmatched_s3)/(len(s2)+len(s3))*100:.1f}%')

In [ ]:
# 10j. Test set quick profile (schema, countries, sizes)
print('\n--- Test set profile ---')
TEST = BASE / 'dataset' / 'test'
test_files = sorted(TEST.glob('*.tsv'))
if test_files:
    for f in test_files:
        mb = f.stat().st_size / 1024 / 1024
        df_test = pd.read_csv(f, sep='\t', dtype=str)
        print(f'  {f.name}: {len(df_test):,} rows, {mb:.1f} MB')
        print(f'    Columns: {list(df_test.columns)}')
        vc = df_test['country'].value_counts()
        for c, n in vc.items():
            print(f'    {c:15s}: {n:>10,} ({n/len(df_test)*100:.1f}%)')
        del df_test
else:
    print('  No test files found (may need to be downloaded)')

In [ ]:
# 10k. Final summary table
print('\n' + '=' * 70)
print('SUMMARY OF KEY FINDINGS')
print('=' * 70)

findings = [
    ('Dataset scale', f'S1={len(s1):,}  S2={len(s2):,}  S3={len(s3):,}  GT={len(gt):,}'),
    ('Countries (train)', str(sorted(s1["country"].unique()))),
    ('Singleton rate', f'{n_singleton/n_total*100:.2f}% ({n_singleton:,} / {n_total:,})'),
    ('Baseline (predict-nothing) F_0.5', f'{baseline_f05:.4f}'),
    ('Country agreement (matched pairs)', f'{agreement_rate*100:.2f}%'),
    ('ID/Position leakage', 'See Section 2 results above'),
    ('Name identical in matched pairs', f'{n_name_identical/max(pair_count,1)*100:.1f}% (of {pair_count:,} pairs sampled)'),
    ('Name script change (Latin↔Indic)', f'{n_name_script_change/max(pair_count,1)*100:.1f}%'),
    ('Address identical in matched pairs', f'{n_addr_identical/max(pair_count,1)*100:.1f}%'),
]

for label, value in findings:
    print(f'  {label:45s}  {value}')

print('\n--- Implications for ML Pipeline ---')
print('  1. Country can likely be used as a hard blocking key (verify exact % above)')
print('  2. Legal suffix stripping is essential but must be country-conditional')
print('  3. Address parsing cannot rely on fixed component order')
print('  4. Script transliteration (Devanagari→Latin) needed for India matches')
print('  5. High distractor ratio in S2/S3 means blocking recall is critical')
print('  6. Token-level similarity (Jaccard, overlap) more robust than char-level for names')
print('  7. F_0.5 heavily penalizes false merges — precision-first strategy needed')